# Data Mining - Lab 9
## Cube space: Data Cube-ийн тооцоолол ба олон хэмжээст шинжилгээ

**Dataset:** `cs_coffee_mini.csv`, `cs_sales.csv`, `cs_dim_item.csv`, `cs_dim_location.csv`, `cs_iceberg_demo.csv`, `cs_closed_demo.csv`, `cs_monthly_item_city.csv`, `cs_loans.csv`

Энэ лаб нь "Cube space: олон хэмжээст өгөгдлийн шинжилгээ" лекцийн агуулгад үндэслэсэн: cuboid, lattice, cell, concept hierarchy, measure, cube-ийн тооцоолол (materialization, iceberg, BUC, closed cell), OLAP архитектур, cube space дахь mining (prediction cube, discovery-driven exploration).

| Файл | Мөр | Тайлбар |
|---|---|---|
| `cs_coffee_mini` | 8 | Лекцийн кофе шопын мини куб (салбар x улирал x бараа) |
| `cs_sales` | 3000 | 2026 оны 1-6 сарын электроникийн борлуулалтын гүйлгээ |
| `cs_dim_item` | 8 | Барааны dimension (төрөл, брэнд, нийлүүлэгчийн төрөл, нэгж үнэ) |
| `cs_dim_location` | 9 | Байршлын dimension (дүүрэг, хот, улс) |
| `cs_iceberg_demo` | 10 | Лекцийн Iceberg/BUC дасгалын бичлэгүүд |
| `cs_closed_demo` | 5 | Closed cell ба quotient cube-ийн жижиг жишээ |
| `cs_monthly_item_city` | 168 | Бараа x сар x хотын борлуулалт (`month = 0` нь 2025 оны 12-р сар, суурь) |
| `cs_loans` | 2370 | Зээлийн шийдвэрийн өгөгдөл (хот x жил), prediction cube-д |

### Lab зорилго
- Cell (base, aggregate, apex), ancestor ба descendant ойлголтыг кодоор ялгах
- Cuboid-ийн lattice, шатлалтай cube-ийн cuboid-ийн тоог тооцох, `GROUP BY CUBE`-ийг хэрэгжүүлэх
- Concept hierarchy (schema ба set grouping) ашиглан roll-up, drill-down хийх
- Distributive, algebraic, holistic measure-ийн нэгтгэлийн дүрмийг шалгах
- Cuboid сонгох, materialization, Iceberg cube, BUC, closed cell, quotient cube-ийг хэрэгжүүлэх
- ROLAP ба MOLAP-ийг харьцуулах
- Discovery-driven exploration ба prediction cube-ээр cube space дотор mining хийх

Даалгавар бүр **TASK** гэсэн гарчигтай. Код бүрэн өгөгдөөгүй: функцийн нэр, хувьсагчийн нэр, `# TODO` маягийн заавар өгсөн тул `...` болон тайлбарын дагуу кодыг **өөрөө бичнэ**. Хариугаа багшид үзүүлж шалгуулна.

## 0) Өгөгдөл унших

Гүйлгээний хүснэгтийг `cs_dim_item`, `cs_dim_location`-тэй холбож нэг өргөн хүснэгт (`df`) гаргана. Үүн дээр сарын түвшинг (`month`) нэмнэ.

In [1]:
import pandas as pd
import numpy as np
import sqlite3, math
from itertools import combinations

sales    = pd.read_csv("cs_sales.csv")
dim_item = pd.read_csv("cs_dim_item.csv")
dim_loc  = pd.read_csv("cs_dim_location.csv")

df = (sales.merge(dim_item, on="item", validate="many_to_one")
           .merge(dim_loc, on="district", validate="many_to_one"))
df["date"]  = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.strftime("%Y-%m")
print(df.shape, df["sales"].sum())
df.head()

(3000, 16) 2602025


,sale_id,date,item,district,customer_group,units,sales,item_key,type,brand,supplier_type,unit_price,location_key,city,country,month
0,1,2026-01-01,Гар,Чаоян,Хувь хүн,4,200,7,Дагалдах хэрэгсэл,Монгол Тех,Дотоодын,50,7,Бээжин,Хятад,2026-01
1,2,2026-01-01,Монитор,Баянзүрх,Бизнес,2,600,2,Компьютер,Dell,Гадаадын,300,3,Улаанбаатар,Монгол,2026-01
2,3,2026-01-01,Гар,Дархан-Уул,Хувь хүн,1,50,7,Дагалдах хэрэгсэл,Монгол Тех,Дотоодын,50,4,Дархан,Монгол,2026-01
3,4,2026-01-01,Хулгана,Хан-Уул,Хувь хүн,5,125,8,Дагалдах хэрэгсэл,Монгол Тех,Дотоодын,25,2,Улаанбаатар,Монгол,2026-01
4,5,2026-01-01,Утас,Чаоян,Хувь хүн,1,400,4,Гар утас ба таблет,Samsung,Гадаадын,400,7,Бээжин,Хятад,2026-01


## 1) Мини куб: Roll-up, slice, dice (лекцийн дасгал)

Кофе шопын борлуулалт (мянган ₮). Dimension: `branch`, `quarter`, `item`. Measure: `sales`.

### TASK 1: Мини кубийн 6 асуулт

1. **Roll-up**: салбар бүрийн нийт борлуулалт.
2. **Slice**: `quarter = Q2` хүснэгтийг (мөр = салбар, багана = бараа) гарга.
3. **Dice**: `branch = УБ` ба `item = Кофе`. Улирал бүрийн утгыг гарга.
4. **Apex**: `(*, *, *)` cell-ийн утга.
5. **Cuboid**: энэ кубэд хэдэн cuboid байх вэ?
6. **Exception**: Q1-ээс Q2 хүртэл хамгийн их хувиар өссөн (салбар, бараа) cell аль вэ?

In [2]:
coffee = pd.read_csv("cs_coffee_mini.csv")

# 1. Roll-up: салбар бүрийн нийт борлуулалт
print(coffee.groupby("branch")["sales"].sum())

# 2. Slice: quarter == "Q2" хүснэгт (мөр = салбар, багана = бараа), pivot_table ашигла
q2 = coffee[coffee["quarter"] == "Q2"].pivot_table(index="branch", columns="item", values="sales", aggfunc="sum")
print(q2)

# 3. Dice: branch == "УБ" ба item == "Кофе", улирал бүрийн утга
print(coffee[(coffee["branch"] == "УБ") & (coffee["item"] == "Кофе")][["quarter", "sales"]])

# 4. Apex-ийн утга ба cuboid-ийн тоо (2 ** n)
print("Apex (*,*,*):", coffee["sales"].sum(), "| cuboid:", 2 ** 3)

# 5. Exception: Q1 -> Q2 өсөлтийг хувиар бодож, хамгийн их өссөн (салбар, бараа)-г ол
pv = coffee.pivot_table(index=["branch", "item"], columns="quarter", values="sales")
pv["growth_%"] = (pv["Q2"] - pv["Q1"]) / pv["Q1"] * 100
print(pv.sort_values("growth_%", ascending=False))
print("Хамгийн их өссөн:", pv["growth_%"].idxmax())


branch
Дархан    200
УБ        400
Name: sales, dtype: int64
item    Кофе  Цай
branch           
Дархан    65   45
УБ       150   70
  quarter  sales
0      Q1    120
2      Q2    150
Apex (*,*,*): 600 | cuboid: 8
quarter         Q1     Q2   growth_%
branch item                         
Дархан Кофе   50.0   65.0  30.000000
УБ     Кофе  120.0  150.0  25.000000
       Цай    60.0   70.0  16.666667
Дархан Цай    40.0   45.0  12.500000
Хамгийн их өссөн: ('Дархан', 'Кофе')


## 2) Cell: base, aggregate, ancestor, descendant

Cell нь cuboid доторх нэг мөр. `*` нь тухайн dimension-ийг нэгтгэсэн (all) гэсэн тэмдэг.

| Cell | Төрөл |
|---|---|
| `(Jan, Chicago, Business)` | base cell (бүх dimension утгатай) |
| `(Jan, *, Business)` | 2-D aggregate cell |
| `(*, *, *)` | apex cell (0-D) |

**Ancestor** (өвөг) нь илүү ерөнхий, **descendant** (үр удам) нь илүү нарийвчилсан cell. `(Jan, *, *)` нь `(Jan, Chicago, Business)`-ийн ancestor.

### TASK 2: Бүх cell-ийг гаргаж ангилах

1. `cube_cells` функцээр мини кубийн **бүх cell** (base, aggregate, apex)-ийг гарга. Нийт cell-ийн тоо `(n₁+1)(n₂+1)(n₃+1)` томьёотой тэнцэх ёстой.
2. `classify` функцийг бөглөж лекцийн Жишээ 3.9-ийн 5 cell-ийг ангил.
3. `is_ancestor` функцээр `(УБ, Q1, Кофе)` cell-ийн ancestor-уудыг, `(УБ, *, *)` cell-ийн descendant-уудыг ол.

In [3]:
dims = ["branch", "quarter", "item"]

def cube_cells(df, dims, measure):
    """Бүх cell (base, aggregate, apex)-ийг нэг DataFrame болгож буцаа.
    Нэгтгэгдсэн dimension-д "*" тавина. Баганууд: dims + [measure]."""
    parts = []
    for k in range(len(dims), -1, -1):
        for sub in combinations(dims, k):
            if k == 0:
                g = pd.DataFrame({measure: [df[measure].sum()]})
            else:
                g = df.groupby(list(sub), as_index=False)[measure].sum()
            for d in dims:
                if d not in sub:
                    g[d] = "*"
            parts.append(g[dims + [measure]])
    return pd.concat(parts, ignore_index=True)

cells = cube_cells(coffee, dims, "sales")
expected = math.prod(coffee[d].nunique() + 1 for d in dims)
print("Нийт cell:", len(cells), "| (n1+1)(n2+1)(n3+1) =", expected)

def classify(cell):
    """cell-ийг "base cell", "apex cell" эсвэл "k-D aggregate cell" гэж ангил."""
    k = sum(v != "*" for v in cell)
    if k == len(cell):
        return "base cell"
    if k == 0:
        return "apex cell"
    return f"{k}-D aggregate cell"

ex = [("Jan", "*", "*"), ("*", "Chicago", "*"), ("Jan", "*", "Business"), ("Jan", "Chicago", "Business"), ("*", "*", "*")]
for c in ex:
    print(c, "->", classify(c))

def is_ancestor(a, b):
    """a нь b-ийн ancestor эсэх: a != b ба a-ийн утга бүр "*" эсвэл b-ийнхтэй тэнцүү."""
    return tuple(a) != tuple(b) and all(x == "*" or x == y for x, y in zip(a, b))

tuples = [tuple(r) for r in cells[dims].itertuples(index=False)]
target = ("УБ", "Q1", "Кофе")
print("Ancestors of", target, ":", [t for t in tuples if is_ancestor(t, target)])
top = ("УБ", "*", "*")
print("Descendants of", top, ":", [t for t in tuples if is_ancestor(top, t)])


Нийт cell: 27 | (n1+1)(n2+1)(n3+1) = 27
('Jan', '*', '*') -> 1-D aggregate cell
('*', 'Chicago', '*') -> 1-D aggregate cell
('Jan', '*', 'Business') -> 2-D aggregate cell
('Jan', 'Chicago', 'Business') -> base cell
('*', '*', '*') -> apex cell
Ancestors of ('УБ', 'Q1', 'Кофе') : [('УБ', 'Q1', '*'), ('УБ', '*', 'Кофе'), ('*', 'Q1', 'Кофе'), ('УБ', '*', '*'), ('*', 'Q1', '*'), ('*', '*', 'Кофе'), ('*', '*', '*')]
Descendants of ('УБ', '*', '*') : [('УБ', 'Q1', 'Кофе'), ('УБ', 'Q1', 'Цай'), ('УБ', 'Q2', 'Кофе'), ('УБ', 'Q2', 'Цай'), ('УБ', 'Q1', '*'), ('УБ', 'Q2', '*'), ('УБ', '*', 'Кофе'), ('УБ', '*', 'Цай')]


## 3) Cuboid-ийн lattice ба тоо

Dimension-ы дэд олонлог бүрээр group-by хийвэл нэг cuboid үүснэ. Эдгээрийн lattice нь data cube.

- Hierarchy-гүй n dimension: **2ⁿ** cuboid
- Hierarchy-тай: **T = ∏ (Lᵢ + 1)**, Lᵢ нь i-р dimension-ийн түвшний тоо, `+1` нь `all`

### TASK 3: Lattice ба cuboid-ийн тоо

1. 4 dimension (`time`, `item`, `location`, `supplier`)-ийн lattice-ийг 0-D-ээс 4-D хүртэл түвшин бүрээр тоол.
2. Лекцийн жишээ: `time` (day < month < year, L = 3), `location` (city < country, L = 2), `item` (type, L = 1). Нийт cuboid хэд вэ?
3. 10 dimension, тус бүр 5 түвшинтэй бол хэд вэ?
4. Манай схем: `time` (day, month, year), `location` (district, city, country), `item` (item, type), `customer_group` (1 түвшин). Нийт cuboid хэд вэ?
5. Яагаад бүгдийг урьдчилан тооцоолох (full materialization) практикт боломжгүй вэ? (үгээр)

In [4]:
dims4 = ["time", "item", "location", "supplier"]

# 1. Lattice-ийн түвшин бүрийн cuboid-ийн тоо (0-D ... 4-D) ба нийт
levels = {k: math.comb(len(dims4), k) for k in range(len(dims4) + 1)}
for k, n in levels.items():
    print(f"{k}-D: {n} cuboid", [c for c in combinations(dims4, k)])
print("Нийт:", sum(levels.values()), "= 2^4 =", 2 ** 4)

# 2. Лекцийн жишээ
hier = {"time": 3, "location": 2, "item": 1}
print("Лекцийн жишээ:", math.prod(L + 1 for L in hier.values()))

# 3. 10 dimension x 5 түвшин
print("10 dim x 5 түвшин:", (5 + 1) ** 10)

# 4. Манай схем
ours = {"time": 3, "location": 3, "item": 2, "customer_group": 1}
print("Манай схем:", math.prod(L + 1 for L in ours.values()))

# 5. Cuboid-ийн тоо dimension болон түвшний тоогоор экспоненциалаар өсдөг (6^10 = 60 сая гаруй).
#    Cuboid бүр олон мөртэй тул бүгдийг тооцоолж хадгалах санах ой, хугацаа хангалтгүй.


0-D: 1 cuboid [()]
1-D: 4 cuboid [('time',), ('item',), ('location',), ('supplier',)]
2-D: 6 cuboid [('time', 'item'), ('time', 'location'), ('time', 'supplier'), ('item', 'location'), ('item', 'supplier'), ('location', 'supplier')]
3-D: 4 cuboid [('time', 'item', 'location'), ('time', 'item', 'supplier'), ('time', 'location', 'supplier'), ('item', 'location', 'supplier')]
4-D: 1 cuboid [('time', 'item', 'location', 'supplier')]
Нийт: 16 = 2^4 = 16
Лекцийн жишээ: 24
10 dim x 5 түвшин: 60466176
Манай схем: 96


## 4) GROUP BY CUBE

SQL-ийн `GROUP BY CUBE (month, city)` нь 2² = 4 cuboid-ийг нэг дор гаргана: `(month, city)`, `(month, NULL)`, `(NULL, city)`, `(NULL, NULL)`. SQL дээр `*` (all)-ийн оронд **NULL** гарна.

### TASK 4: CUBE-ийг өөрөө хэрэгжүүлэх

1. `cube_groupby` функц бич: dimension-ы бүх дэд олонлог тус бүрээр бүлэглэж, нэгтгэгдсэн dimension-д `NaN` (NULL) тавь.
2. `df` дээр `["month", "city"]`-ээр CUBE хий. Cuboid тус бүрд хэдэн мөр гарсныг тоол.
3. Apex мөр (бүх dimension NULL) нийт борлуулалттай тэнцэх ёстой. Шалга.
4. SQLite нь `CUBE`-г дэмждэггүй. `UNION ALL`-аар ижил үр дүнг SQL-ээр гаргаж, pandas-ийн үр дүнтэй харьцуул.

In [5]:
def cube_groupby(df, dims, measure, agg="sum"):
    """GROUP BY CUBE: dimension-ы бүх дэд олонлог тус бүрээр бүлэглэ.
    Нэгтгэгдсэн dimension-д np.nan (NULL) тавь. Баганууд: dims + [measure]."""
    parts = []
    for k in range(len(dims), -1, -1):
        for sub in combinations(dims, k):
            if k == 0:
                g = pd.DataFrame({measure: [df[measure].agg(agg)]})
            else:
                g = df.groupby(list(sub), as_index=False)[measure].agg(agg)
            for d in dims:
                if d not in sub:
                    g[d] = np.nan
            g["cuboid"] = "(" + ", ".join(sub) + ")" if sub else "()"
            parts.append(g[dims + [measure, "cuboid"]])
    return pd.concat(parts, ignore_index=True)

# 1. ["month", "city"]-ээр CUBE хий; cuboid тус бүрийн мөрийн тоог гарга
cube = cube_groupby(df, ["month", "city"], "sales")
print(cube.groupby("cuboid", sort=False).size())

# 2. Apex мөр нийт борлуулалттай тэнцэх эсэхийг шалга
apex = cube[cube["month"].isna() & cube["city"].isna()]["sales"].iloc[0]
print("Apex:", apex, "| нийт:", df["sales"].sum(), "| тэнцүү:", apex == df["sales"].sum())

# 3. SQLite дээр ижил үр дүнг UNION ALL-аар гарга, pandas-ийн үр дүнтэй харьцуул
con = sqlite3.connect(":memory:")
df[["month", "city", "sales"]].to_sql("sales_table", con, index=False)
q = """
SELECT month, city, SUM(sales) AS sales FROM sales_table GROUP BY month, city
UNION ALL SELECT month, NULL, SUM(sales) FROM sales_table GROUP BY month
UNION ALL SELECT NULL, city, SUM(sales) FROM sales_table GROUP BY city
UNION ALL SELECT NULL, NULL, SUM(sales) FROM sales_table
"""
sql = pd.read_sql(q, con)
key = ["month", "city"]
a = cube[key + ["sales"]].fillna("NULL").sort_values(key).reset_index(drop=True)
b = sql.fillna("NULL").sort_values(key).reset_index(drop=True)
print("SQL мөр:", len(sql), "| pandas мөр:", len(cube), "| ижил:", a.astype(str).equals(b.astype(str)))


cuboid
(month, city)    36
(month)           6
(city)            6
()                1
dtype: int64
Apex: 2602025 | нийт: 2602025 | тэнцүү: True
SQL мөр: 49 | pandas мөр: 49 | ижил: True


## 5) Concept hierarchy: roll-up ба drill-down

Concept hierarchy нь доод түвшний ойлголтыг дээд түвшний ойлголтод холбох дараалал. Roll-up бол модоор дээш (дүүрэг -> хот -> улс), drill-down бол доош.

| Төрөл | Тайлбар | Жишээ |
|---|---|---|
| Schema hierarchy | Атрибутын бүрэн эсвэл хэсэгчилсэн дараалал | `day < month < quarter < year`; `day < week` |
| Set grouping hierarchy | Утгыг мужаар бүлэглэнэ | `price`: `($0..$200]`, `($200..$400]` |

Интервалд `(` нь эхлэх утгыг оруулахгүй, `]` нь төгсгөлийн утгыг оруулна.

### TASK 5: Шатлал бүрээр нэгтгэх

1. **Байршил**: борлуулалтыг `country`, дараа нь `city`, дараа нь `district`-ээр гарга. Улс бүрийн нийлбэр хот бүрийнхийн нийлбэртэй тэнцэх ёстой.
2. **Drill-down**: `Монгол` улсаас эхлэж хот руу, хамгийн их хотоос дүүрэг рүү задал.
3. **Хугацаа**: `quarter` (Q1, Q2) баганыг нэм. Улирлаар нэгтгэ.
4. **Хэсэгчилсэн дараалал**: `week` нь `month`-ийн дотор бүрэн багтдаггүй. Нэгээс олон сард хамаарсан ISO долоо хоногийг тоол.
5. **Set grouping**: барааны `unit_price`-ийг 200 нэгжийн интервалаар `pd.cut`-аар бүлэглэ (баруун тал хаалттай). Зөвхөн хэдэн барааны үнэ 200 дээр яг тохиолдож, ямар интервалд орохыг шалга. Дараа нь 100 нэгжийн интервалаар нарийвчил (drill-down).

In [6]:
# 1. Байршил: country, city, district-ээр борлуулалтыг гарга; нийлбэрүүд тэнцэх эсэхийг шалга
by_country = df.groupby("country")["sales"].sum()
by_city = df.groupby(["country", "city"])["sales"].sum()
by_district = df.groupby(["country", "city", "district"])["sales"].sum()
print(by_country, by_city, sep="\n\n")
print("Улс = хотын нийлбэр:", by_city.groupby(level=0).sum().equals(by_country))
print("Хот = дүүргийн нийлбэр:", by_district.groupby(level=[0, 1]).sum().equals(by_city))

# 2. Drill-down: Монгол -> хот -> хамгийн их хотоос дүүрэг рүү
mn_city = by_city.loc["Монгол"].sort_values(ascending=False)
print(mn_city)
top_city = mn_city.idxmax()
print(f"{top_city}-ийн дүүргүүд:")
print(by_district.loc[("Монгол", top_city)].sort_values(ascending=False))

# 3. df["quarter"] баганыг нэм ("Q1", "Q2") ба улирлаар нэгтгэ
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)
print(df.groupby("quarter")["sales"].sum())

# 4. ISO долоо хоног нэгээс олон сард хамаарч байгаа эсэх
df["week"] = df["date"].dt.isocalendar().week
wk = df.groupby("week")["month"].nunique()
print("Нийт долоо хоног:", len(wk), "| 2+ сард хамаарсан:", (wk > 1).sum(), [int(w) for w in wk[wk > 1].index])

# 5. Set grouping: unit_price-ийг 200, дараа нь 100 нэгжийн интервалаар
items = dim_item.copy()
items["band200"] = pd.cut(items["unit_price"], bins=range(0, 1001, 200), right=True)
items["band100"] = pd.cut(items["unit_price"], bins=range(0, 1001, 100), right=True)
print(items[["item", "unit_price", "band200", "band100"]])
print("Үнэ = 200 бараа:", items.loc[items["unit_price"] == 200, ["item", "band200"]].values.tolist())
d = df.merge(items[["item", "band200", "band100"]], on="item")
print(d.groupby("band200", observed=True).agg(items=("item", "nunique"), sales=("sales", "sum")))
print(d.groupby("band100", observed=True).agg(items=("item", "nunique"), sales=("sales", "sum")))


country
Монгол    1887750
Хятад      714275
Name: sales, dtype: int64

country  city       
Монгол   Дархан          333200
         Улаанбаатар    1131800
         Ховд            149875
         Эрдэнэт         272875
Хятад    Бээжин          472975
         Шанхай          241300
Name: sales, dtype: int64
Улс = хотын нийлбэр: True
Хот = дүүргийн нийлбэр: True
city
Улаанбаатар    1131800
Дархан          333200
Эрдэнэт         272875
Ховд            149875
Name: sales, dtype: int64
Улаанбаатар-ийн дүүргүүд:
district
Сүхбаатар    429450
Хан-Уул      393525
Баянзүрх     308825
Name: sales, dtype: int64
quarter
Q1    1283550
Q2    1318475
Name: sales, dtype: int64
Нийт долоо хоног: 27 | 2+ сард хамаарсан: 4 [5, 9, 14, 18]
      item  unit_price     band200     band100
0  Ноутбук         800  (600, 800]  (700, 800]
1  Монитор         300  (200, 400]  (200, 300]
2  Принтер         200    (0, 200]  (100, 200]
3     Утас         400  (200, 400]  (300, 400]
4   Таблет         500  (400, 600] 

## 6) Measure: distributive, algebraic, holistic

| Төрөл | Жишээ | Хэсгийн үр дүнгээс нийт тооцох |
|---|---|---|
| Distributive | `sum`, `count`, `min`, `max` | Хэсгийн үр дүнг нэгтгэхэд бүтнээр нь бодсонтой адил |
| Algebraic | `avg`, `stddev`, `min_N`, `max_N` | Тогтмол M ширхэг distributive функцээс |
| Holistic | `median`, `mode`, `rank` | Боломжгүй, бүх утгыг хадгална |

### TASK 6: Хэмжүүрийн нэгтгэлийн шалгалт

1. Лекцийн жишээ: Салбар A = 10, 20, 30; Салбар B = 40, 50. `sum`, `count`, `avg`, `median`-ийг A, B, бүгд гэж тус бүр бод. Хэсгээс нэгтгэж болох, болохгүйг тогтоо.
2. Бодит өгөгдөл: барааны `type` бүрийг нэг хэсэг гэж үзэж `sales`-ийн **дундаж**-ыг (a) бүх мөрөөр, (b) `type` бүрийн дундажуудын дундажаар, (c) `type` бүрийн `(sum, count)` хосоор бод. Аль нь зөв бэ? Яагаад (b) буруу байна вэ?
3. **Стандарт хазайлт**-ыг `type` бүрийн `(n, Σx, Σx²)` хосоос бод. Бүтнээр нь бодсонтой тэнцэх ёстой (`ddof = 0`).
4. **median**-ыг `type`-ийн median-уудын median-аар бодоод бүтнийхтэй харьцуул.
5. **max_N** (top 3): `type` бүрийн top 3-ыг нэгтгэж дахин top 3 авбал бүтэн өгөгдлийн top 3-тай тэнцэх үү?

In [7]:
# 1. Лекцийн жишээ: A = [10, 20, 30], B = [40, 50]
A, B = np.array([10, 20, 30]), np.array([40, 50])
AB = np.concatenate([A, B])
tbl = pd.DataFrame({f: [g(A), g(B), g(AB)] for f, g in
                    {"sum": np.sum, "count": len, "avg": np.mean, "median": np.median}.items()},
                   index=["A", "B", "бүгд"])
print(tbl)
print("sum хэсгээс:", A.sum() + B.sum(), "| count:", len(A) + len(B),
      "| avg (sum, count)-оос:", (A.sum() + B.sum()) / (len(A) + len(B)),
      "| median-уудын median:", np.median([np.median(A), np.median(B)]), "!= бодит", np.median(AB))

# 2. type-ийг хэсэг гэж үзэх
g = df.groupby("type")["sales"]
a = df["sales"].mean()
b = g.mean().mean()
c = g.sum().sum() / g.count().sum()
print(f"(a) {a:.4f}  (b) {b:.4f}  (c) {c:.4f}")
# (b) буруу: хэсэг бүрийн мөрийн тоо ялгаатай тул жинлэхгүй дундажлах нь алдаа гаргана.

# 3. Стандарт хазайлт (n, Σx, Σx²)-аас
agg = g.agg(n="count", s="sum", s2=lambda x: (x ** 2).sum())
N, S, S2 = agg["n"].sum(), agg["s"].sum(), agg["s2"].sum()
std_parts = math.sqrt(S2 / N - (S / N) ** 2)
print("std хэсгээс:", std_parts, "| бүтэн:", df["sales"].std(ddof=0), "| ижил:", np.isclose(std_parts, df["sales"].std(ddof=0)))

# 4. median
print("median-уудын median:", g.median().median(), "| бүтэн median:", df["sales"].median())

# 5. max_N (top 3)
top3_parts = sorted(int(v) for v in np.concatenate([x.nlargest(3).values for _, x in g]))[::-1][:3]
top3_full = df["sales"].nlargest(3).tolist()
print("хэсгээс top3:", top3_parts, "| бүтэн:", top3_full, "| ижил:", list(top3_parts) == top3_full)


      sum  count   avg  median
A      60      3  20.0    20.0
B      90      2  45.0    45.0
бүгд  150      5  30.0    30.0
sum хэсгээс: 150 | count: 5 | avg (sum, count)-оос: 30.0 | median-уудын median: 32.5 != бодит 30.0
(a) 867.3417  (b) 921.3431  (c) 867.3417
std хэсгээс: 858.2343162547289 | бүтэн: 858.234316254729 | ижил: True
median-уудын median: 900.0 | бүтэн median: 600.0
хэсгээс top3: [4000, 4000, 4000] | бүтэн: [4000, 4000, 4000] | ижил: True


## 7) Query-д cuboid сонгох

Query ирэхэд хамгийн бага хэмжээтэй, асуултыг хариулж чадах materialized cuboid-ийг сонгоно.

1. Query-ийн GROUP BY ба WHERE-ээс шаардлагатай түвшнийг тогтооно.
2. Hierarchy-гээр тухайн түвшин эсвэл **илүү нарийвчилсан** түвшний cuboid-ийг хайна (хойно roll-up хийж болно).
3. Хамгийн бага өртөгтэй (мөрийн тоо хамгийн цөөн) cuboid-ийг сонгоно.

Cuboid-ийг `(time түвшин, location түвшин, item түвшин)` гэж тэмдэглэнэ, `None` нь `all`. Жишээ нь `("month", "city", None)`.

### TASK 7: Cuboid сонгогч

Дараах cuboid-ууд materialized: `("day","district","item")`, `("month","city","item")`, `("month","city",None)`, `("month","country","item")`, `("year","country",None)`.

1. `can_answer`: materialized cuboid нь query-г хариулж чадах нөхцөл. Dimension бүрт materialized түвшин query-ийн түвшингээс **нарийн эсвэл тэнцүү** байх ёстой.
2. `size`: cuboid-ийн мөрийн тоо (өгөгдлөөс).
3. Дараах query-үүдэд хариулж чадах cuboid-уудыг жагсааж, хамгийн бага хэмжээтэйг сонго:
   - Q1: `("month", "country", None)`
   - Q2: `("year", "city", "type")`
   - Q3: `(None, None, None)` (apex)
4. Q2-ыг `("month","city",None)` cuboid яагаад хариулж чадахгүй вэ? (үгээр)

In [8]:
df["day"]  = df["date"].dt.strftime("%Y-%m-%d")
df["year"] = df["date"].dt.year

LEVELS = {"time": ["day", "month", "year"],
          "location": ["district", "city", "country"],
          "item": ["item", "type"]}

def rank(dim, level):
    return LEVELS[dim].index(level) if level else len(LEVELS[dim])    # None = all, хамгийн ерөнхий

def can_answer(mat, q):
    """materialized cuboid mat нь query q-г хариулж чадах уу?
    Dimension бүрт mat-ийн түвшин q-ийн түвшингээс нарийн эсвэл тэнцүү байх ёстой."""
    return all(rank(d, m) <= rank(d, x) for d, m, x in zip(LEVELS, mat, q))

def size(c):
    """Cuboid-ийн мөрийн тоо (өгөгдлөөс). None-ийг алгас, бүгд None бол 1."""
    cols = [lvl for lvl in c if lvl is not None]
    return 1 if not cols else df.groupby(cols).ngroups

materialized = [("day", "district", "item"), ("month", "city", "item"), ("month", "city", None),
                ("month", "country", "item"), ("year", "country", None)]
queries = [("month", "country", None), ("year", "city", "type"), (None, None, None)]

for m in materialized:
    print(m, size(m))
for q in queries:
    ok = [(m, size(m)) for m in materialized if can_answer(m, q)]
    best = min(ok, key=lambda t: t[1])
    print(f"\nQuery {q}: хариулж чадах = {ok}\n  -> сонгох: {best}")

# 4. ("month","city",None) нь item-ийг аль хэдийн all болгож нэгтгэсэн тул type түвшин рүү
#    буцаж задлах (drill-down) боломжгүй. Нарийн түвшнээс ерөнхий рүү л roll-up хийж болно.


('day', 'district', 'item') 2616
('month', 'city', 'item') 288
('month', 'city', None) 36
('month', 'country', 'item') 96
('year', 'country', None) 2

Query ('month', 'country', None): хариулж чадах = [(('day', 'district', 'item'), 2616), (('month', 'city', 'item'), 288), (('month', 'city', None), 36), (('month', 'country', 'item'), 96)]
  -> сонгох: (('month', 'city', None), 36)

Query ('year', 'city', 'type'): хариулж чадах = [(('day', 'district', 'item'), 2616), (('month', 'city', 'item'), 288)]
  -> сонгох: (('month', 'city', 'item'), 288)

Query (None, None, None): хариулж чадах = [(('day', 'district', 'item'), 2616), (('month', 'city', 'item'), 288), (('month', 'city', None), 36), (('month', 'country', 'item'), 96), (('year', 'country', None), 2)]
  -> сонгох: (('year', 'country', None), 2)


## 8) Materialization: Full, No, Partial

| Арга | Тодорхойлолт | Онцлог |
|---|---|---|
| Full | Бүх cuboid-ийг урьдчилан тооцно | Query хурдан, хадгалалт их |
| No | Нэгтгэлийг хэрэгтэй үед тооцно | Хадгалалт бага, query удаан |
| Partial | Түгээмэл cuboid-уудыг хадгална | Хурд ба хадгалалтын тэнцвэр |

### TASK 8: 3 dimension-ийн cube (item, city, month)

1. 8 cuboid-ийн мөрийн тоог гарга. **Full** materialization-ий нийт хадгалалт, **No** materialization-ий хадгалалт (зөвхөн base cuboid) хэд вэ?
2. Query бүр өөрийн cuboid-ийг хариулахад, хариулж чадах хамгийн жижиг materialized cuboid-ийг уншина гэж үзвэл 8 cuboid бүрийн query-ийн **дундаж өртөг** хэд вэ? (Full ба No-д тус тусад нь.)
3. **Partial**: base-ээс гадна 2 cuboid сонгож (нийт 3), дундаж өртгийг хамгийн бага болгох жагсаалтыг greedy аргаар ол.
4. Partial-ийн хадгалалт ба дундаж өртгийг Full, No-той харьцуул. Лекцийн "зөвхөн 3 cuboid хадгалбал ямар materialization вэ?" асуултад хариул.

In [9]:
D3 = ["item", "city", "month"]

# 1. 8 cuboid-ийн мөрийн тоо
cuboids = [c for k in range(3, -1, -1) for c in combinations(D3, k)]
sizes = {c: (df.groupby(list(c)).ngroups if c else 1) for c in cuboids}
for c, s in sizes.items():
    print(c or "()", s)
base = tuple(D3)
print("Full хадгалалт:", sum(sizes.values()), "| No хадгалалт:", sizes[base])

# 2. cost, avg_cost
def cost(q, mat):
    return min(sizes[m] for m in mat if set(q) <= set(m))

def avg_cost(mat):
    return np.mean([cost(q, mat) for q in cuboids])

full, no = cuboids, [base]
print("Дундаж өртөг Full:", avg_cost(full), "| No:", avg_cost(no))

# 3. Greedy: base-ээс гадна 2 cuboid
chosen = [base]
for step in range(2):
    best = min((c for c in cuboids if c not in chosen), key=lambda c: avg_cost(chosen + [c]))
    chosen.append(best)
    print(f"Алхам {step + 1}: {best} нэмэв -> дундаж өртөг {avg_cost(chosen):.1f}")

# 4. Харьцуулалт
cmp = pd.DataFrame({"хадгалалт": [sum(sizes[c] for c in m) for m in (full, no, chosen)],
                    "дундаж өртөг": [avg_cost(m) for m in (full, no, chosen)]},
                   index=["Full", "No", "Partial"])
print(cmp)
# Зөвхөн 3 cuboid хадгалбал энэ нь Partial materialization.


('item', 'city', 'month') 288
('item', 'city') 48
('item', 'month') 48
('city', 'month') 36
('item',) 8
('city',) 6
('month',) 6
() 1
Full хадгалалт: 441 | No хадгалалт: 288
Дундаж өртөг Full: 55.125 | No: 288.0
Алхам 1: ('city', 'month') нэмэв -> дундаж өртөг 162.0
Алхам 2: ('item', 'city') нэмэв -> дундаж өртөг 102.0
         хадгалалт  дундаж өртөг
Full           441        55.125
No             288       288.000
Partial        372       102.000


## 9) Iceberg cube, antimonotonicity ба BUC

**Iceberg cube**: `COUNT(*) >= min_sup` нөхцөл хангасан cell-үүдийг л хадгалсан cube.

```
compute cube sales_iceberg as
select month, city, customer_group, count(*)
from salesInfo cube by month, city, customer_group
having count(*) >= min_sup
```

**Antimonotonicity**: нэг cell босго хангахгүй бол түүнээс нарийвчилсан бүх descendant ч хангахгүй. **BUC** (Bottom-Up Computation) нь ингэж **pruning** хийж iceberg cube-ийг үр ашигтай тооцдог.

### TASK 9: Лекцийн дасгал (min_sup = 3) ба BUC

`cs_iceberg_demo.csv` нь 10 бичлэгтэй (item, city): Ном УБ x4, Ном Дархан x1, Дэвтэр УБ x3, Дэвтэр Дархан x2.

1. `(item, city)`-ийн COUNT-ийг гарга. `min_sup = 3` үед Iceberg cube-д аль cell-үүд үлдэх вэ? (CUBE-ийн бүх cuboid-ийг оруул.)
2. `buc` функцийг бөглөж ажиллуул. Аль cell-үүдийн цаашдын салбарыг **prune** хийсэн бэ?
3. Том өгөгдөл: `["month", "city", "customer_group", "item"]`-ээр `min_sup = 30` үед BUC-ийн үр дүн CUBE-ээр бүгдийг тооцоод шүүсэнтэй ижил эсэхийг шалга. Хэдэн cell үлдэв, хэдэн partition хянагдав, нийт cell хэд байв?
4. Antimonotonicity-г шалга: iceberg cell бүрийн ancestor бүр мөн iceberg-д орсон эсэх.

In [10]:
demo = pd.read_csv("cs_iceberg_demo.csv")
min_sup = 3

# 1. (item, city)-ийн COUNT; cube_groupby-аар бүх cuboid-ийг гаргаж min_sup-аар шүү
print(demo.value_counts(["item", "city"]))
demo["cnt"] = 1
cnt_cube = cube_groupby(demo, ["item", "city"], "cnt")
iceberg = cnt_cube[cnt_cube["cnt"] >= min_sup]
print(iceberg.fillna("*")[["item", "city", "cnt"]])

def buc(df, dims, min_sup):
    """BUC: (cell, count) жагсаалт, prune хийсэн cell-ийн жагсаалт, хянасан partition-ийн тоог буцаа."""
    out, pruned = [], []
    checked = 0

    def rec(part, prefix, start):
        nonlocal checked
        checked += 1
        if len(part) < min_sup:
            pruned.append(tuple(prefix))
            return
        out.append((tuple(prefix), len(part)))
        for i in range(start, len(dims)):
            for val, sub in part.groupby(dims[i], sort=True):
                p = list(prefix)
                p[i] = val
                rec(sub, p, i + 1)

    rec(df, ["*"] * len(dims), 0)
    return out, pruned, checked

# 2. buc(demo, ["item", "city"], 3)
res, pruned, checked = buc(demo, ["item", "city"], 3)
print("Iceberg cell:", res)
print("Prune хийсэн:", pruned, "| хянасан partition:", checked)

# 3. Том өгөгдөл
d4 = ["month", "city", "customer_group", "item"]
res4, pruned4, checked4 = buc(df, d4, 30)
df["cnt"] = 1
full4 = cube_groupby(df, d4, "cnt")
ice4 = full4[full4["cnt"] >= 30]
set_buc = {(c, n) for c, n in res4}
set_cube = {(tuple("*" if pd.isna(v) else v for v in r[:-1]), r[-1])
            for r in ice4[d4 + ["cnt"]].itertuples(index=False)}
print("BUC == CUBE шүүлт:", set_buc == set_cube)
print("Үлдсэн cell:", len(res4), "| хянасан partition:", checked4, "| нийт cell:", len(full4))

# 4. Antimonotonicity
ice_cells = {c for c, _ in res4}
def ancestors(cell):
    idx = [i for i, v in enumerate(cell) if v != "*"]
    for k in range(len(idx)):
        for keep in combinations(idx, k):
            yield tuple(v if i in keep else "*" for i, v in enumerate(cell))
print("Бүх ancestor iceberg-д байгаа:", all(a in ice_cells for c in ice_cells for a in ancestors(c)))


item    city  
Ном     УБ        4
Дэвтэр  УБ        3
        Дархан    2
Ном     Дархан    1
Name: count, dtype: int64
     item    city  cnt
1  Дэвтэр      УБ    3
3     Ном      УБ    4
4  Дэвтэр       *    5
5     Ном       *    5
6       *  Дархан    3
7       *      УБ    7
8       *       *   10
Iceberg cell: [(('*', '*'), 10), (('Дэвтэр', '*'), 5), (('Дэвтэр', 'УБ'), 3), (('Ном', '*'), 5), (('Ном', 'УБ'), 4), (('*', 'Дархан'), 3), (('*', 'УБ'), 7)]
Prune хийсэн: [('Дэвтэр', 'Дархан'), ('Ном', 'Дархан')] | хянасан partition: 9


BUC == CUBE шүүлт: True
Үлдсэн cell: 310 | хянасан partition: 1136 | нийт cell: 1636
Бүх ancestor iceberg-д байгаа: True


## 10) Closed cell, quotient cube, shell fragment

- **Closed cell**: өөрөөсөө илүү нарийвчилсан, **яг ижил суурь нүднүүдийг (base cells) хамарсан** өөр cell байхгүй cell.
- **Quotient cube**: зөвхөн closed cell-үүдээс бүрдсэн шахсан cube.
- **Shell fragment**: dimension-уудыг жижиг бүлэгт хувааж зөвхөн тэдгээрийн cuboid-ийг урьдчилан тооцох арга.

### TASK 10: Closed cell олох

`cs_closed_demo.csv` нь 5 суурь мөртэй: Дархан Q1 Кофе, Дархан Q1 Цай, УБ Q2 Кофе, УБ Q2 Цай, УБ Q3 Цай.

1. `cube_cells`-ээр (TASK 2) бүх non-empty cell-ийг гарга. Нийт хэдэн cell байна вэ?
2. Cell бүрийн **cover** (хамарсан суурь мөрийн индексийн олонлог)-ийг тооц.
3. Closed cell нь: өөрийн cover-тэй **яг ижил cover-тэй descendant байхгүй** cell. Closed cell-үүдийг ол.
4. `(Дархан, *, *)` closed уу? Яагаад? Түүнийг төлөөлөх closed cell аль вэ?
5. Quotient cube-д хэдэн cell үлдэх вэ? Шахалтын харьцаа хэд вэ?
6. Shell fragment: 4 dimension-ийг `{time, item}` ба `{location, supplier}` хоёр бүлэгт хувааж, бүлэг доторх cuboid (apex-гүй)-ийг л тооцвол хэдэн cuboid тооцох вэ? Бүтэн cube-тэй (apex-гүй 15) харьцуул.

In [11]:
cd = pd.read_csv("cs_closed_demo.csv")
dims_c = ["branch", "quarter", "item"]

# 1. Бүх non-empty cell
cc = cube_cells(cd, dims_c, "sales")
cell_list = [tuple(r) for r in cc[dims_c].itertuples(index=False)]
print("Non-empty cell:", len(cell_list))

# 2. covers(cell)
def covers(cell):
    m = np.ones(len(cd), dtype=bool)
    for d, v in zip(dims_c, cell):
        if v != "*":
            m &= (cd[d] == v).values
    return frozenset(np.where(m)[0])

# 3. is_closed(cell)
def is_closed(cell):
    cv = covers(cell)
    return not any(is_ancestor(cell, o) and covers(o) == cv for o in cell_list)

# 4. Closed cell-ийн жагсаалт
closed = [c for c in cell_list if is_closed(c)]
for c in closed:
    print(c, sorted(int(x) for x in covers(c)))
dk = ("Дархан", "*", "*")
print(dk, "closed:", is_closed(dk))
rep = [o for o in closed if covers(o) == covers(dk)]
print("Төлөөлөх closed cell:", rep)

# 5. Quotient cube
print(f"Quotient cube: {len(closed)} cell / {len(cell_list)} -> шахалт {len(closed) / len(cell_list):.2%}")

# 6. Shell fragment
groups = [["time", "item"], ["location", "supplier"]]
shell = sum(2 ** len(g) - 1 for g in groups)
print("Shell fragment cuboid:", shell, "| бүтэн cube (apex-гүй):", 2 ** 4 - 1)


Non-empty cell: 25
('Дархан', 'Q1', 'Кофе') [0]
('Дархан', 'Q1', 'Цай') [1]
('УБ', 'Q2', 'Кофе') [2]
('УБ', 'Q2', 'Цай') [3]
('УБ', 'Q3', 'Цай') [4]
('Дархан', 'Q1', '*') [0, 1]
('УБ', 'Q2', '*') [2, 3]
('УБ', '*', 'Цай') [3, 4]
('УБ', '*', '*') [2, 3, 4]
('*', '*', 'Кофе') [0, 2]
('*', '*', 'Цай') [1, 3, 4]
('*', '*', '*') [0, 1, 2, 3, 4]
('Дархан', '*', '*') closed: False
Төлөөлөх closed cell: [('Дархан', 'Q1', '*')]
Quotient cube: 12 cell / 25 -> шахалт 48.00%
Shell fragment cuboid: 6 | бүтэн cube (apex-гүй): 15


## 11) OLAP серверийн архитектур: ROLAP ба MOLAP

| Архитектур | Хадгалалт | Нэгтгэл |
|---|---|---|
| ROLAP | Relational хүснэгт | SQL |
| MOLAP | Олон хэмжээст массив | Массивын индекс, тэнхлэгээр нэгтгэх |
| HOLAP | Хослуулсан | Нэгтгэсэн хэсэг MOLAP, дэлгэрэнгүй хэсэг ROLAP |

### TASK 11: Ижил асуулга, хоёр архитектур

1. **ROLAP**: SQLite-д `fact` хүснэгт үүсгэж `GROUP BY city, month`-ээр SUM гарга.
2. **MOLAP**: `item x city x month` dense numpy массив үүсгэ (`np.add.at`). `item` тэнхлэгийг roll-up хий (`sum(axis=0)`).
3. Хоёр үр дүн ижил эсэхийг шалга.
4. **Sparsity**: `district x item x day x customer_group` dimension-ийн dense массивын нийт cell-ийн тоо, **хоосон биш** cell-ийн тоо, хоосон биш cell-ийн хувийг гарга. Яагаад олон dimension-тэй cube-д MOLAP-ийн dense массив хэмжээ зарцуулагддаг вэ? HOLAP яагаад үүнийг засдаг вэ? (үгээр)

In [12]:
# 1. ROLAP
con = sqlite3.connect(":memory:")
df[["item", "city", "month", "sales"]].to_sql("fact", con, index=False)
rolap = pd.read_sql("SELECT city, month, SUM(sales) AS sales FROM fact GROUP BY city, month ORDER BY city, month", con)
print(rolap.head())

# 2. MOLAP
it = sorted(df["item"].unique()); ct = sorted(df["city"].unique()); mn = sorted(df["month"].unique())
cube3 = np.zeros((len(it), len(ct), len(mn)))
np.add.at(cube3, (df["item"].map({v: i for i, v in enumerate(it)}),
                  df["city"].map({v: i for i, v in enumerate(ct)}),
                  df["month"].map({v: i for i, v in enumerate(mn)})), df["sales"])
molap = cube3.sum(axis=0)

# 3. Харьцуулалт
r = rolap.pivot(index="city", columns="month", values="sales").reindex(index=ct, columns=mn).fillna(0).values
print("ROLAP == MOLAP:", np.allclose(r, molap))

# 4. Sparsity
dd = ["district", "item", "day", "customer_group"]
total = math.prod(df[c].nunique() for c in dd)
nonempty = df.groupby(dd).ngroups
print(f"Dense cell: {total:,} | хоосон биш: {nonempty:,} | {nonempty / total:.2%}")
# Dimension нэмэгдэх тусам dense массивын хэмжээ үржвэрээр өсөх боловч бодит хослолын ихэнх нь хоосон.
# HOLAP нь нягт, нэгтгэсэн хэсгийг MOLAP-д, сийрэг дэлгэрэнгүй хэсгийг ROLAP-д хадгалдаг.


     city    month  sales
0  Бээжин  2026-01  82050
1  Бээжин  2026-02  83275
2  Бээжин  2026-03  62775
3  Бээжин  2026-04  82300
4  Бээжин  2026-05  73800
ROLAP == MOLAP: True
Dense cell: 39,096 | хоосон биш: 2,829 | 7.24%


## 12) Discovery-driven exploration

Сая сая cell-ээс сонирхолтойг гараар хайх боломжгүй. Хүлээгдэж буй утгаас ихээр зөрсөн cell-ийг стандартчилсан хазайлтаар (жишээ нь |z| > 2.5) автоматаар тэмдэглэнэ.

- **SelfExp**: cell өөрөө хэр онцгой
- **InExp**: cell-ийн доор (drill-down) онцгой утга нуугдсан эсэх
- **PathExp**: drill-down зам бүрийн онцгой байдал

### TASK 12: SelfExp ба InExp

`cs_monthly_item_city.csv`-д 4 бараа x 6 хот x сар (0 = 2025 оны 12-р сар).

1. Барааны сарын борлуулалтын өөрчлөлтийг (%) гарга: `chg` (бараа x сар 1-6). Лекцийн хүснэгттэй адил эсэхийг шалга.
2. **SelfExp**: хүлээгдэх утгыг мөрийн дундаж + баганын дундаж - нийт дундаж гэж үзэж, үлдэгдэл (residual)-ийг стандартчил. Хамгийн онцгой 4 cell, мөн |z| > 2 болох cell-ийн тоог ол (жижиг хүснэгт тул босгыг 2 гэж авна).
3. **InExp**: cell бүрийг хот руу drill-down хийж, хотын өөрчлөлт нь улсын өөрчлөлтөөс (`chg`) хэр зөрөхийг стандартчил. Бараа x сар бүрт хотуудын хамгийн их |z|-ийг ол.
4. SelfExp өндөр боловч InExp нам, мөн SelfExp нам боловч InExp өндөр cell байна уу? Тэдгээрийг тайлбарла. (Лекцийн Принтер 4-р сар жишээ)

In [13]:
mo = pd.read_csv("cs_monthly_item_city.csv")

# 1. nat ба chg
nat = mo.pivot_table(index="item", columns="month", values="sales", aggfunc="sum")
chg = (nat.pct_change(axis=1) * 100).loc[:, 1:6].round(1)
print(chg)

# 2. SelfExp
exp = chg.mean(axis=1).values[:, None] + chg.mean(axis=0).values[None, :] - chg.values.mean()
resid = chg - exp
z_self = resid / resid.values.std()
s = z_self.stack().abs().sort_values(ascending=False)
print("SelfExp top 4:\n", s.head(4))
print("|z| > 2:", (s > 2).sum())

# 3. InExp
cpv = mo.pivot_table(index=["item", "city"], columns="month", values="sales", aggfunc="sum")
cchg = (cpv.pct_change(axis=1) * 100).loc[:, 1:6]
diff = cchg - chg.reindex(cchg.index.get_level_values(0)).values
z_in = diff / diff.values.std()
in_exp = z_in.abs().groupby(level=0).max()
print("InExp (бараа x сар, хотын хамгийн их |z|):\n", in_exp.round(2))
item_i, month_i = in_exp.stack().idxmax()
city_i = z_in.xs(item_i, level=0)[month_i].abs().idxmax()
print("Хамгийн онцгой InExp:", item_i, "сар", month_i, "->", city_i)

# 4. Хоёрыг харьцуулах
both = pd.DataFrame({"SelfExp": z_self.abs().stack(), "InExp": in_exp.stack()})
print("SelfExp өндөр, InExp нам:\n", both[(both.SelfExp > 2) & (both.InExp < 2)])
print("SelfExp нам, InExp өндөр:\n", both[(both.SelfExp < 1) & (both.InExp > 2)])


month      1    2     3    4     5    6
item                                   
Ноутбук  2.0  3.0   1.0  4.0   2.0  3.0
Принтер -1.0  0.1   0.9 -2.0   0.1  0.9
Утас     1.0  2.0 -14.0  2.0   1.0  3.0
Чихэвч   2.9  2.2   3.9  1.0  22.0  2.0
SelfExp top 4:
 item     month
Чихэвч   5        2.884844
Утас     3        2.270141
Ноутбук  5        1.213176
Принтер  3        1.194918
dtype: float64
|z| > 2: 2
InExp (бараа x сар, хотын хамгийн их |z|):
 month       1     2     3      4     5     6
item                                        
Ноутбук  0.51  0.65  0.52   0.28  0.32  0.57
Принтер  0.59  1.27  0.77  10.59  0.34  0.47
Утас     0.56  0.43  0.33   0.60  0.76  0.57
Чихэвч   0.84  1.09  0.48   0.43  0.56  0.60
Хамгийн онцгой InExp: Принтер сар 4 -> Ховд
SelfExp өндөр, InExp нам:
                SelfExp     InExp
item   month                    
Утас   3      2.270141  0.334774
Чихэвч 5      2.884844  0.562820
SelfExp нам, InExp өндөр:
                 SelfExp      InExp
item    month   

## 13) Prediction cube

Cell бүрт тухайн cell-ийн өгөгдлөөр сургасан загварын **accuracy**-г measure болгон хадгална. Хоёр хэлбэр: (1) загварын accuracy, (2) attribute-ийн predictiveness, өөрөөр хэлбэл attribute-тэй ба гүй загварын accuracy-ийн зөрүү.

`cs_loans.csv`: зээлийн шийдвэр (`approved`). Dimension: `city`, `year`. Feature: `age`, `income`, `debt_ratio`, `gender`.

### TASK 13: Зээлийн prediction cube

1. (`city`, `year`) cell бүр дээр `age`, `income`, `debt_ratio`-оор logistic regression-ийг 5-fold cross validation-ээр сургаж accuracy-г гарга (`StandardScaler` ашигла).
2. Cell бүрт `gender`-ийг нэмж accuracy-г дахин гарга. Зөрүү (`gender_gain`) нь gender-ийн predictiveness.
3. Accuracy-ийн (хот x жил) хүснэгтийг гарга. Загвар хамгийн сул хот аль вэ? Яагаад (cell дэх өгөгдлийн хэмжээ, шуугианыг бод)?
4. Аль хотод `gender`-ийг нэмэхэд accuracy тогтвортой өсч байна вэ? Ховд-д зөрүү яагаад тогтворгүй байна вэ? Энэ нь fairness-ийн ямар сэрэмжлүүлэг вэ? (үгээр)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

loans = pd.read_csv("cs_loans.csv")

def cell_acc(g, feats):
    """Нэг cell (g)-ийн өгөгдөл дээр feats-ээр logistic regression-ийг
    5-fold cross validation-ээр сургаж дундаж accuracy-г буцаа (StandardScaler-тай pipeline)."""
    ...

# 1. (city, year) cell бүрт age, income, debt_ratio-оор accuracy
# 2. gender нэмсэн accuracy ба зөрүү (gender_gain)
# 3. pc DataFrame үүсгэж, accuracy (хот x жил) ба gender_gain-ийн хүснэгтийг pivot-оор гарга

## Тайлан бичих заавар

Тайланг **`Lab9_Тайлан_загвар.docx`** (Лабораторийн тайлангийн загвар №9) файл дээр бичнэ. Загварын 5 хэсэг дээр энэ ноутбукийн агуулга дараах байдлаар тусна.

| Тайлангийн хэсэг | Энэ лабораторид юу бичих вэ? |
|---|---|
| 1. Зорилго | Лабын зорилго (cell, cuboid, hierarchy, measure; cube-ийн тооцоолол; cube space дахь mining). Хүрсэн эсэхээ 5-р хэсэгт бичнэ |
| 2. Онол | Лекцийн слайдын ойлголтуудыг өөрийн үгээр товч (cell, cuboid, ∏(Lᵢ + 1), concept hierarchy, measure-ийн 3 төрөл, materialization, Iceberg, BUC, closed cell, ROLAP/MOLAP, SelfExp/InExp, prediction cube) |
| 3. Аргачлал | Ноутбукийн **чухал 3-5 код** (жишээ нь `cube_groupby`, `buc`, `is_closed`, SelfExp/InExp, `cell_acc`) ба тус бүр ямар онолыг хэрхэн хэрэгжүүлснийг 2-3 өгүүлбэрээр |
| 4. Үр дүн | TASK 1-13-ын үр дүнгийн хүснэгт (загварт бэлэн), тохиромжтой график, хүснэгтийн зураг |
| 5. Дүгнэлт | Юу ойлгосон, юу хэцүү байсан, яаж шийдсэн. Доорх асуултуудад хариулна |
| Хавсралт | A. `Lab9_CubeSpace.ipynb`, `cs_*.csv`; B. git log; C. Python хувилбар, сангууд; D. GitHub холбоос |

### Дүгнэлт хэсгийн асуултууд
1. Аль даалгавар хамгийн хэцүү байсан бэ? Яаж шийдсэн бэ?
2. Antimonotonicity-г өөрийн үгээр тайлбарла. BUC-д pruning яагаад зөв байдаг вэ?
3. Cube-ийг бүгдийг нь тооцохын оронд Iceberg, closed cell, partial materialization ашиглах нь яагаад хэрэгтэй вэ? (3-5 өгүүлбэр)
4. Өөрийн салбарт (борлуулалт, сургалт, эрүүл мэнд гэх мэт) discovery-driven exploration-г хэрхэн хэрэглэх вэ?